# Storage block $\mathbf{M}$ — inertance & end corrections on the jump elements

The acoustic operator is $\mathbf A(\omega)=\mathbf J_{\mathrm{alg}}+i\omega\mathbf M+\mathbf P(\omega)+\mathbf S(\omega)$.
The [`helmholtz_resonator`](helmholtz_resonator.ipynb) notebook introduced the **compliance** face of $\mathbf M$ through the `cavity`.
This notebook adds its **dual** — the **inertance** — and the **end corrections**, generalized to the ordinary *jump* elements (area changes, losses, manifolds).

A lengthless jump stores nothing.  But a real component has axial extent, so its dropped $\frac{\mathrm d}{\mathrm dt}\!\int_V\mathbf U\,\mathrm dV$ term returns as

$$
\underbrace{+\frac{\ell_i A_i}{\overline c_i^{\,2}}\,\widehat p_i'}_{\text{compliance (mass row)}}
\qquad\text{and}\qquad
\underbrace{-\frac{L_{\mathrm{eff}}}{A}\,\widehat{\dot m}'}_{\text{inertance (pressure row)}},
\qquad L_{\mathrm{eff}}=\ell_{\mathrm{geom}}+\delta_{\mathrm{end}} .
$$

The inputs are **physically relatable**: a *length* per side on the passages (`l_up`, `l_down`, `end_correction`), a *volume* on the manifolds.
See `scratch/inertance-end-correction-theory.md`.

In [ ]:
import warnings
import numpy as np
import plotly.graph_objects as go

from fns.shell import Network
from fns.elements import catalog as cat
from fns.thermo.configure import perfect_gas
from fns.perturbation import perturbation_response, build_acoustic_blocks
from fns.solver.control import states_table
from fns.derive import ES_C, ES_AREA
from fns.plotting import use_fns_theme

_ = use_fns_theme()

#  --- calorically perfect air at rest ---
GAMMA, R = 1.4, 287.0
GAS = perfect_gas(R, GAMMA)
P0, T0 = 101325.0, 300.0
C0 = np.sqrt(GAMMA * R * T0)  # sound speed [m/s]

#  --- side-branch resonator geometry ---
A_MAIN, L_MAIN = 3.0e-3, 0.05  # main duct
A_NECK, L_NECK = 5.0e-4, 0.02  # neck
V_CAV = 1.0e-3  # cavity / plenum volume
A_NECK_RADIUS = np.sqrt(A_NECK / np.pi)

FREQS = np.linspace(50.0, 1200.0, 1200)


def helmholtz_f0(c, An, V, Leff):
    """Lumped Helmholtz frequency f0 = c*sqrt(An/(V*Leff))/2pi."""
    return c * np.sqrt(An / (V * Leff)) / (2.0 * np.pi)

## Part 1 — inertance is the reactive dual of the neck

The Helmholtz resonator needs a neck **inertance**.  Until now that came only from a neck *duct* (carried in $\mathbf P(\omega)$).
The storage block now supplies the same inertance as a length on a *lengthless* element:
a `linear_resistance` (here with $R=0$) carrying half-lengths `l_up = l_down = \ell/2` is a neck of geometric length $\ell$ — inertance $\ell$ **and** the matching compliance $\ell A$ — but stamped into $\mathbf M$.

We build the same side-branch resonator two ways and overlay the transmission loss:

```
inlet ─ duct ─ junction ─ duct ─ outlet
                  └─ NECK ─ cavity
```

In [ ]:
def side_branch(neck, vol=V_CAV):
    """inlet-duct-junction-duct-outlet, with a junction-NECK-cavity side branch.

    `neck` is a callable (net, tee_node) -> (mid_node) that adds the neck element(s)
    between the tee and a freshly added cavity; returns (Solution, in_edge, out_edge).
    """
    net = Network(GAS, p_ref=P0, T_ref=T0, mdot_ref=1.0)
    inlet = net.add(cat.mass_flow_inlet(0.0, T0))
    d_in = net.add(cat.duct(L_MAIN))
    tee = net.add(cat.junction())
    d_out = net.add(cat.duct(L_MAIN))
    outlet = net.add(cat.pressure_outlet(P0, T0))
    e_in = net.connect(inlet, d_in, A_MAIN)
    net.connect(d_in, tee, A_MAIN)
    net.connect(tee, d_out, A_MAIN)
    e_out = net.connect(d_out, outlet, A_MAIN)
    neck(net, tee, vol)
    sol = net.solve()
    assert sol.converged
    return sol, e_in, e_out


def neck_as_duct(net, tee, vol):
    neck = net.add(cat.duct(L_NECK))
    cav = net.add(cat.cavity(vol))
    net.connect(tee, neck, A_NECK)
    net.connect(neck, cav, A_NECK)


def neck_as_inertance(net, tee, vol, leff=L_NECK):
    #  a lengthless element carrying the neck length -> inertance via M (l split -> + compliance)
    neck = net.add(cat.linear_resistance(0.0, l_up=leff / 2, l_down=leff / 2))
    cav = net.add(cat.cavity(vol))
    net.connect(tee, neck, A_NECK)
    net.connect(neck, cav, A_NECK)


def tl(sol, e_in, e_out):
    resp = perturbation_response(sol.problem, sol.x, FREQS)
    with warnings.catch_warnings():  # the side branch is a branch point; the transmission dip is still physical
        warnings.simplefilter("ignore")
        tau = resp.acoustic_scattering_matrix(e_in, e_out)[:, 1, 0]
    return -20.0 * np.log10(np.abs(tau))


sol_d, ei, eo = side_branch(neck_as_duct)
sol_i, _, _ = side_branch(neck_as_inertance)
tl_duct = tl(sol_d, ei, eo)
tl_inert = tl(sol_i, ei, eo)
f0 = helmholtz_f0(C0, A_NECK, V_CAV, L_NECK)
print(f"analytic f0 = {f0:.1f} Hz")
print(f"neck as duct (P)        : peak {FREQS[np.argmax(tl_duct)]:.1f} Hz")
print(f"neck as inertance (M)   : peak {FREQS[np.argmax(tl_inert)]:.1f} Hz")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=FREQS, y=tl_duct, name="neck as duct  —  P(ω)",
                         line=dict(color="black", width=4), opacity=0.35))
fig.add_trace(go.Scatter(x=FREQS, y=tl_inert, name="neck as inertance  —  iωM",
                         line=dict(color="#1f77b4", width=1.8)))
fig.add_vline(x=f0, line=dict(color="#d62728", width=1, dash="dot"),
              annotation_text="f0", annotation_position="top")
fig.update_layout(title="Inertance via M reproduces the neck duct",
                  xaxis_title="Frequency, Hz", yaxis_title="Transmission Loss, dB",
                  width=820, height=460)
fig.show()

**Result.** The two routes resonate at the *same* Helmholtz frequency — the inertance the duct carries in its phase $\mathbf P(\omega)$ is exactly the leading-order storage $i\omega\,L_{\mathrm{eff}}/A$ the lengthless element now stamps into $\mathbf M$ (theory.md §12.5).
The lumped inertance is purely reactive, so its peak is a touch sharper than the (distributed) duct's near resonance.

We can read the storage coefficients straight out of $\mathbf M$:

In [ ]:
blocks = build_acoustic_blocks(sol_i.problem, sol_i.x)
est = states_table(sol_i.problem, sol_i.x)
prob = sol_i.problem
ns = int(prob.n_solve)
#  the neck element is node 5 (inlet, d_in, tee, d_out, outlet, NECK, cavity)
n_neck = 5
base = int(prob.row_ptr[n_neck])
e0, e1 = int(prob.col_edge[base]), int(prob.col_edge[base + 1])
r_mass = int(prob.node_row_ptr[n_neck])
Mco = blocks.M.tocoo()
M = {(int(r), int(c)): complex(v) for r, c, v in zip(Mco.row, Mco.col, Mco.data)}
A_n, c_n = float(est[ES_AREA, e0]), float(est[ES_C, e0])
print("storage entries on the neck element:")
print(f"  compliance  l_up*A/c^2 = {M[(r_mass, ns*e0+1)].real:.3e}   (expected {(L_NECK/2)*A_n/c_n**2:.3e})")
print(f"  inertance   |L_eff/A|  = {abs(M[(r_mass+1, ns*e0+0)]):.3e}   (expected {L_NECK/A_n:.3e})")

## Part 2 — end corrections

A neck does not stop at its geometric mouth: it entrains a near-field slug of fluid, the **end correction** $\delta_{\mathrm{end}}\approx0.85\,a$ (flanged).
The effective length is $L_{\mathrm{eff}}=\ell_{\mathrm{geom}}+\delta_{\mathrm{end}}$, and it lowers the resonance — using the bare geometric length over-predicts $\overline f_0$.

The `end_correction` argument adds straight into the inertance.  We sweep it and watch the peak move onto the corrected $\overline f_0=\overline c\sqrt{A_n/(V\,L_{\mathrm{eff}})}/2\pi$.

In [ ]:
fig = go.Figure()
palette = ["#1f77b4", "#ff7f0e", "#2ca02c"]
print("  delta/a    L_eff [mm]   f0 analytic   f_peak FNS")
for col, frac in zip(palette, [0.0, 0.85, 1.7]):
    delta = frac * A_NECK_RADIUS
    leff = L_NECK + delta

    def neck(net, tee, vol, leff=leff):
        neck_el = net.add(cat.linear_resistance(0.0, l_up=L_NECK / 2, l_down=L_NECK / 2, end_correction=delta))
        cav = net.add(cat.cavity(vol))
        net.connect(tee, neck_el, A_NECK)
        net.connect(neck_el, cav, A_NECK)

    sol, ei, eo = side_branch(neck)
    curve = tl(sol, ei, eo)
    f0c = helmholtz_f0(C0, A_NECK, V_CAV, leff)
    fpk = FREQS[int(np.argmax(curve))]
    label = "bare (δ=0)" if frac == 0 else f"δ = {frac:.2f} a"
    fig.add_trace(go.Scatter(x=FREQS, y=curve, name=f"{label}  (f0={f0c:.0f})", line=dict(color=col, width=1.8)))
    fig.add_vline(x=f0c, line=dict(color=col, width=1, dash="dot"))
    print(f"  {frac:5.2f}    {leff*1e3:8.2f}    {f0c:9.1f}    {fpk:9.1f}")

fig.update_layout(title="End correction lengthens the neck and lowers f0",
                  xaxis_title="Frequency, Hz", yaxis_title="Transmission Loss, dB",
                  width=820, height=460)
fig.show()

**Result.** Each end correction lengthens $L_{\mathrm{eff}}$ and the FNS peak tracks the corrected $\overline f_0$ exactly.
The bare geometric neck sits highest; a flanged $\delta=0.85\,a$ pulls the resonance down by ~20 %.
This is the modeling input that closes the residual offset seen in the pure-geometric Helmholtz notebook.

## Part 3 — a plenum is a compliance: the manifold `volume`

For a chamber the natural input is a **volume**, not a length.  A `junction(volume=V)` gives the manifold the compliance $C=V/\overline c^{\,2}$ on its (common) pressure — *a junction with a volume is a cavity with through-flow*.

To show the duality we replace the side-branch `cavity(V)` with a `junction(volume=V)` capped by a `wall` (a 2-port manifold whose second port is blocked) and confirm the **same** resonance:

```
… junction ─ neck duct ─ junction(volume=V) ─ wall
```

In [ ]:
def neck_then_plenum(net, tee, vol):
    neck = net.add(cat.duct(L_NECK))
    plenum = net.add(cat.junction(volume=vol))  # the chamber compliance lives here
    wall = net.add(cat.wall())                   # caps the manifold (a dead volume)
    net.connect(tee, neck, A_NECK)
    net.connect(neck, plenum, A_NECK)
    net.connect(plenum, wall, A_NECK)


sol_p, ei, eo = side_branch(neck_then_plenum)
tl_plenum = tl(sol_p, ei, eo)

fig = go.Figure()
fig.add_trace(go.Scatter(x=FREQS, y=tl_duct, name="cavity(V)  compliance",
                         line=dict(color="black", width=4), opacity=0.35))
fig.add_trace(go.Scatter(x=FREQS, y=tl_plenum, name="junction(volume=V) + wall",
                         line=dict(color="#2ca02c", width=1.8)))
fig.add_vline(x=f0, line=dict(color="#d62728", width=1, dash="dot"),
              annotation_text="f0", annotation_position="top")
fig.update_layout(title="Manifold volume reproduces the cavity compliance",
                  xaxis_title="Frequency, Hz", yaxis_title="Transmission Loss, dB",
                  width=820, height=460)
fig.show()
print(f"cavity peak           : {FREQS[np.argmax(tl_duct)]:.1f} Hz")
print(f"junction-volume peak  : {FREQS[np.argmax(tl_plenum)]:.1f} Hz")

**Takeaway.** The storage block $\mathbf M$ is now general:

| input | element family | stores |
|-------|----------------|--------|
| `l_up`, `l_down`, `end_correction` | area changes, `loss`, `linear_resistance` | per-port **compliance** + series **inertance** |
| `volume` | `junction`, `splitter`, `cavity` | chamber **compliance** |

Every coefficient is read off the same per-port machinery (`col_edge`), defaults to zero (so existing networks are untouched), and is assembled into $\mathbf A=\mathbf J_{\mathrm{alg}}+i\omega\mathbf M+\mathbf P+\mathbf S$ with no element-specific acoustic code.